In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

In [ ]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess
from matplotlib.collections import PathCollection

In [ ]:
import sys

sys.path.append('..')

from pyfuncs.io import load_full_adata, add_ensembl_ids
from pyfuncs.dropletQC import classify_empty_and_damaged
from pyfuncs.general import preprocessing_adata_sub
from pyfuncs.plot_functions import magma, set_plotting_style, plot_volcano, plot_cell_stats, plot_gene_stats, savefig
from pyfuncs.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [ ]:
from pyfuncs.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition, reprocess_in_place
from pyfuncs.cell_types import DICT_MARKERS_MAJOR_POPULATIONS, DICT_MARKERS_FAP, DICT_MARKERS_KRANOCYTE, DICT_MARKERS_SATELLITE, DICT_MARKERS_TENO, DICT_RENAMING, PALETTE_CELL_TYPE

In [ ]:
from pyfuncs.io import ambient_fraction_per_gene
from pyfuncs.trajectory import (
    normalize_velocyto_layers,
    run_velocity,
    paga_report,
    paga_stability,
    plot_velocity,
    run_phate
)

In [ ]:
from datetime import date
TODAY = str(date.today())

DATA_DIR = f"{BASE_DIR}/data/ARAUZO_03/"
FIG_DIR = f"{BASE_DIR}/figures/{TODAY}/"
RESULTS_DIR = f"{BASE_DIR}/results/{TODAY}/"

CELL_TYPE="cell_type"
SEED = 10

In [ ]:
GRUPO = "cell_type"     # la partición sobre la que se define PAGA
LOTE  = "gsm"

# Globinas: 45% del transcriptoma en algunas muestras, y 100% spliced.
# Fuera de velocity_genes sí o sí.
HB = ["Hbb-bs", "Hba-a1", "Hba-a2", "Hbb-bt", "Alas2", "Ahsp", "Bpgm"]

# Adata loading

In [ ]:
adata = sc.read(f"{DATA_DIR}/processed_adatas/AA_inhouse_dataset_processed.h5ad")
adata_FAP = sc.read(f"{DATA_DIR}/processed_adatas/AA_inhouse_dataset_processed_FAPs.h5ad")

## FAPs

In [ ]:
adata_FAP_velo = adata_FAP.copy()

reprocess_in_place(adata_FAP_velo, integrate=True)         
adata_FAP_velo.var["ambient_fraction"] = ambient_fraction_per_gene(adata_FAP_velo)

normalize_velocyto_layers(adata_FAP_velo, verbose=True)
run_velocity(adata_FAP_velo, use_rep="X_harmony", normalize_su=False, correct_su=False,
             ambient_max=0.2, n_jobs=20)

adata_FAP_velo.var.loc[adata_FAP_velo.var_names.isin(HB), "velocity_genes"] = False
print("genes de velocity tras quitar globinas:", int(adata_FAP_velo.var["velocity_genes"].sum()))

# velocity_genes cambió, así que el grafo y el pseudotiempo hay que rehacerlos
scv.tl.velocity_graph(adata_FAP_velo, n_jobs=20)
scv.tl.velocity_pseudotime(adata_FAP_velo)
scv.tl.velocity_confidence(adata_FAP_velo)
print("confianza media:", round(float(adata_FAP_velo.obs["velocity_confidence"].mean()), 3))


In [ ]:
adata_FAP_velo.obs[GRUPO] = adata_FAP_velo.obs[GRUPO].astype("category").cat.remove_unused_categories()

rep = paga_report(adata_FAP_velo, GRUPO, use_time_prior="velocity_pseudotime")
# Si tienes una variable ordinal externa (día, estadio, dosis, un score):
# rep = paga_report(a, GRUPO, external_key="timepoint")

aristas = rep["aristas"]
print("\nlas 8 aristas más fuertes:")
print(aristas.head(8)[["direccion", "flujo_neto", "conectividad",
                       "delta_pseudotiempo"]].round(3).to_string(index=False))

In [ ]:
adata_FAP_velo = run_phate(adata_FAP_velo, seed=SEED)

In [ ]:
fig, ax = plt.subplots(1, 1)
plot_velocity(adata_FAP_velo, basis="umap", color=GRUPO, legend_fontsize=10,
              legend_loc="right", alpha=1, s=10, arrow_size=4,
              arrow_length=10, title="1bc", ax=ax)
savefig(fig=fig, filename=f"2AA_velocyto_UMAP_FAP", fig_dir=FIG_DIR)
# El threshold es una decisión de dibujo: mira la tabla de aristas y elige uno
# que deje ver la estructura sin inventarla. Prueba varios y quédate con el que
# declares.

fig, ax = plt.subplots(1,1)

scv.pl.umap(adata_FAP_velo, color=GRUPO, ax=ax, show=False, legend_loc="right")
scv.pl.paga(adata_FAP_velo, basis="umap", color=GRUPO, size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, threshold=0.15, 
            title=f"PAGA dirigido (threshold=0.15)", show=False, ax=ax)

node_collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

node_collections[-1].set_edgecolor("black")
node_collections[-1].set_linewidth(0.5)
savefig(fig=fig, filename=f"2AA_PAGA_UMAP_FAP", fig_dir=FIG_DIR)


In [ ]:
fig, ax = plt.subplots(1, 1)
plot_velocity(adata_FAP_velo, basis="phate", color=GRUPO, legend_fontsize=10,
              legend_loc="right", alpha=1, s=10, arrow_size=4,
              arrow_length=10, title="1bc", ax=ax)
savefig(fig=fig, filename=f"2AA_velocyto_PHATE_FAP", fig_dir=FIG_DIR)

# El threshold es una decisión de dibujo: mira la tabla de aristas y elige uno
# que deje ver la estructura sin inventarla. Prueba varios y quédate con el que
# declares.

fig, ax = plt.subplots(1,1)
scv.pl.phate(adata_FAP_velo, color=GRUPO, ax=ax, show=False, legend_loc="right")
scv.pl.paga(adata_FAP_velo, basis="phate", color=GRUPO, size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, threshold=0.15, 
            title=f"PAGA dirigido (threshold=0.15)", show=False, ax=ax)

node_collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

node_collections[-1].set_edgecolor("black")
node_collections[-1].set_linewidth(0.5)
savefig(fig=fig, filename=f"2AA_PAGA_PHATE_FAP", fig_dir=FIG_DIR)


In [ ]:
estab = paga_stability(adata_FAP_velo, GRUPO, n_boot=10, frac=0.8, n_jobs=20, seed=0)

firmes = estab[(estab["estabilidad"] >= 0.8) & estab["estabilidad"].notna()]
print("\naristas reportables:")
print(firmes[["a", "b", "flujo_completo", "estabilidad"]].round(3).to_string(index=False))

## FAPs + TNMD + SAT (ENDO are removed)

In [ ]:
adata_velo = adata.copy()
adata_velo = adata_velo[adata_velo.obs["cell_type"] != "ENDO"]
sc.pp.filter_genes(adata_velo, min_counts = 1)

reprocess_in_place(adata_velo, integrate=True)         
adata_velo.var["ambient_fraction"] = ambient_fraction_per_gene(adata_FAP_velo)

normalize_velocyto_layers(adata_velo, verbose=True)
run_velocity(adata_velo, use_rep="X_harmony", normalize_su=False, correct_su=False,
             ambient_max=0.2, n_jobs=20)

adata_velo.var.loc[adata_velo.var_names.isin(HB), "velocity_genes"] = False
print("genes de velocity tras quitar globinas:", int(adata_velo.var["velocity_genes"].sum()))

# velocity_genes cambió, así que el grafo y el pseudotiempo hay que rehacerlos
scv.tl.velocity_graph(adata_velo, n_jobs=20)
scv.tl.velocity_pseudotime(adata_velo)
scv.tl.velocity_confidence(adata_velo)
print("confianza media:", round(float(adata_velo.obs["velocity_confidence"].mean()), 3))


In [ ]:
adata_velo.obs[GRUPO] = adata_velo.obs[GRUPO].astype("category").cat.remove_unused_categories()

rep = paga_report(adata_velo, GRUPO, use_time_prior="velocity_pseudotime")
# Si tienes una variable ordinal externa (día, estadio, dosis, un score):
# rep = paga_report(a, GRUPO, external_key="timepoint")

aristas = rep["aristas"]
print("\nlas 8 aristas más fuertes:")
print(aristas.head(8)[["direccion", "flujo_neto", "conectividad",
                       "delta_pseudotiempo"]].round(3).to_string(index=False))

In [ ]:
adata_velo = run_phate(adata_velo, seed=SEED)

In [ ]:
fig, ax = plt.subplots(1, 1)
plot_velocity(adata_velo, basis="umap", color=GRUPO, legend_fontsize=10,
              legend_loc="right", alpha=1, s=10, arrow_size=4,
              arrow_length=10, title="1bc", ax=ax)
savefig(fig=fig, filename=f"2AA_velocyto_UMAP_FAP-sat-tnmd", fig_dir=FIG_DIR)
# El threshold es una decisión de dibujo: mira la tabla de aristas y elige uno
# que deje ver la estructura sin inventarla. Prueba varios y quédate con el que
# declares.

fig, ax = plt.subplots(1,1)

scv.pl.umap(adata_velo, color=GRUPO, ax=ax, show=False, legend_loc="right")
scv.pl.paga(adata_velo, basis="umap", color=GRUPO, size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, threshold=0.15, 
            title=f"PAGA dirigido (threshold=0.15)", show=False, ax=ax)

node_collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

node_collections[-1].set_edgecolor("black")
node_collections[-1].set_linewidth(0.5)
savefig(fig=fig, filename=f"2AA_PAGA_UMAP_FAP-sat-tnmd", fig_dir=FIG_DIR)


In [ ]:
fig, ax = plt.subplots(1, 1)
plot_velocity(adata_velo, basis="phate", color=GRUPO, legend_fontsize=10,
              legend_loc="right", alpha=1, s=10, arrow_size=4,
              arrow_length=10, title="1bc", ax=ax)
savefig(fig=fig, filename=f"2AA_velocyto_PHATE_FAP-sat-tnmd", fig_dir=FIG_DIR)

# El threshold es una decisión de dibujo: mira la tabla de aristas y elige uno
# que deje ver la estructura sin inventarla. Prueba varios y quédate con el que
# declares.

fig, ax = plt.subplots(1,1)
scv.pl.phate(adata_velo, color=GRUPO, ax=ax, show=False, legend_loc="right")
scv.pl.paga(adata_velo, basis="phate", color=GRUPO, size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, threshold=0.1, 
            title=f"PAGA dirigido (threshold=0.15)", show=False, ax=ax)

node_collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

node_collections[-1].set_edgecolor("black")
node_collections[-1].set_linewidth(0.5)
savefig(fig=fig, filename=f"2AA_PAGA_PHATE_FAP-sat-tnmd", fig_dir=FIG_DIR)


In [ ]:
estab = paga_stability(adata_velo, GRUPO, n_boot=10, frac=0.8, n_jobs=20, seed=0)

firmes = estab[(estab["estabilidad"] >= 0.8) & estab["estabilidad"].notna()]
print("\naristas reportables:")
print(firmes[["a", "b", "flujo_completo", "estabilidad"]].round(3).to_string(index=False))